# Lab Exercise 6 — Multilayer Perceptron (MLP) Using Artificial Neural Networks

**Course:** MAI511-2 Advanced Machine Learning  
**Problem type:** Binary classification  
**Dataset:** Wisconsin Diagnostic Breast Cancer (WDBC), provided by scikit-learn  
**Aim:** Design, preprocess, train, evaluate, and analyse an MLP, including its architecture and trainable parameters.

> This notebook follows the supplied lab brief. The dataset is a real-world diagnostic dataset bundled with scikit-learn, so no external download is required. The target is used for an educational classification exercise, not clinical decision-making.

## 1. Problem definition and dataset

Predict whether a breast mass is **malignant** or **benign** from numeric measurements computed from digitized images of fine needle aspirates.

- **Input features:** 30 numeric measurements (for example, mean radius, texture, perimeter, area, and concavity).
- **Target:** diagnosis (`0 = malignant`, `1 = benign`).
- **Task:** binary classification.

The dataset has no categorical predictors, and the selected scikit-learn version contains no missing values. The pipeline nevertheless includes median imputation so it remains robust if missing values occur in future input data.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
import sklearn
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             roc_curve)
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("scikit-learn:", sklearn.__version__)

## 2. Load data and inspect it

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
# sklearn target: 0=malignant, 1=benign
y = dataset.target.astype(int).copy()

df = X.copy()
df["diagnosis"] = y.map({0: "malignant", 1: "benign"})
display(df.head())
print("Dataset shape:", df.shape)
print("\\nTarget counts:")
display(df["diagnosis"].value_counts().rename_axis("diagnosis").to_frame("count"))
print("\\nMissing values:", int(X.isna().sum().sum()))

### Dataset overview visualizations

In [ ]:
fig = px.histogram(df, x="diagnosis", color="diagnosis", title="Class distribution",
                   color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(template="plotly_white")
fig.show()

corr = X.corr(numeric_only=True)
fig = px.imshow(corr, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Feature correlation heatmap", aspect="auto")
fig.update_layout(height=750, template="plotly_white")
fig.show()

fig = px.box(df, x="diagnosis", y="mean radius", color="diagnosis",
             title="Mean radius by diagnosis", points="outliers")
fig.update_layout(template="plotly_white")
fig.show()

## 3. Preprocessing and train/test split

1. Split first, using stratification to preserve class proportions.
2. Fit the imputer and scaler **only on the training set** to avoid data leakage.
3. Apply the fitted transformations to the test set.
4. Standardize features because MLP optimization is sensitive to feature scale.

There are no categorical input features in this dataset, so categorical encoding is not applicable.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train = preprocessor.fit_transform(X_train_raw).astype("float32")
X_test = preprocessor.transform(X_test_raw).astype("float32")
y_train_np = y_train.to_numpy().astype("float32")
y_test_np = y_test.to_numpy().astype("int32")

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Input neurons:", X_train.shape[1])

## 4. MLP architecture design

Baseline architecture:

- **Input:** 30 features.
- **Hidden layer 1:** 64 neurons, ReLU.
- **Hidden layer 2:** 32 neurons, ReLU.
- **Output:** 1 neuron, sigmoid.
- **Loss:** binary cross-entropy.
- **Optimizer:** Adam.

ReLU is a computationally simple non-linear activation that helps learn complex feature interactions. Sigmoid maps the output to a probability in `[0, 1]`, appropriate for binary classification. A threshold of 0.5 is used for class labels.

In [ ]:
def build_mlp(input_dim, activation="relu", hidden=(64, 32), dropout=0.0):
    model = Sequential(name=f"MLP_{activation}_{'_'.join(map(str, hidden))}")
    model.add(Input(shape=(input_dim,), name="input"))
    for i, units in enumerate(hidden, start=1):
        model.add(Dense(units, activation=activation, name=f"hidden_{i}"))
        if dropout > 0:
            model.add(Dropout(dropout, name=f"dropout_{i}"))
    model.add(Dense(1, activation="sigmoid", name="output"))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_mlp(X_train.shape[1])
model.summary()

## 5. Train the model

Early stopping monitors validation loss and restores the best weights. A validation split is drawn from the training data; the held-out test set is not used during training.

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
history = model.fit(
    X_train, y_train_np,
    validation_split=0.20,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

### Training curves

In [ ]:
hist = pd.DataFrame(history.history)
hist["epoch"] = np.arange(1, len(hist) + 1)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Accuracy", "Loss"))
fig.add_trace(go.Scatter(x=hist.epoch, y=hist.accuracy, name="Training accuracy"), row=1, col=1)
fig.add_trace(go.Scatter(x=hist.epoch, y=hist.val_accuracy, name="Validation accuracy"), row=1, col=1)
fig.add_trace(go.Scatter(x=hist.epoch, y=hist.loss, name="Training loss"), row=1, col=2)
fig.add_trace(go.Scatter(x=hist.epoch, y=hist.val_loss, name="Validation loss"), row=1, col=2)
fig.update_layout(template="plotly_white", height=420, title="Learning curves")
fig.show()

## 6. Model summary and trainable parameter mathematics

For a Dense layer:

\[
\text{Parameters} = (\text{input connections} \times \text{neurons}) + \text{biases}
\]

Equivalently, for `n_in` inputs and `n_out` neurons:

\[
P = (n_{in} \times n_{out}) + n_{out}
\]

The input layer has no trainable parameters. Dropout layers (if present) also have no trainable parameters. The output Dense layer has one sigmoid neuron.

In [ ]:
def dense_parameter_table(keras_model):
    rows = []
    previous_units = None
    for layer in keras_model.layers:
        if isinstance(layer, Dense):
            units = layer.units
            input_units = int(layer.kernel.shape[0])
            weights = input_units * units
            biases = units
            rows.append({
                "Layer": layer.name,
                "Input connections": input_units,
                "Neurons": units,
                "Weights": weights,
                "Biases": biases,
                "Formula": f"({input_units} × {units}) + {units}",
                "Trainable parameters": weights + biases
            })
    return pd.DataFrame(rows)

param_table = dense_parameter_table(model)
display(param_table)
print("Calculated total:", int(param_table["Trainable parameters"].sum()))
print("Keras total params:", model.count_params())
print("Trainable params:", sum(int(np.prod(v.shape)) for v in model.trainable_weights))
print("Non-trainable params:", sum(int(np.prod(v.shape)) for v in model.non_trainable_weights))

### Architecture diagram (interactive Plotly)

In [ ]:
layer_names = ["Input (30)", "Dense (64)", "Dense (32)", "Output (1)"]
fig = go.Figure()
for i, name in enumerate(layer_names):
    fig.add_trace(go.Scatter(x=[i], y=[0], mode="markers+text",
                             marker=dict(size=48, color=px.colors.qualitative.Bold[i]),
                             text=[name], textposition="bottom center",
                             showlegend=False))
for i in range(len(layer_names)-1):
    fig.add_annotation(x=i+1, y=0, ax=i, ay=0, xref="x", yref="y", axref="x", ayref="y",
                       text="", showarrow=True, arrowhead=3, arrowsize=1.5)
fig.update_layout(title="MLP architecture", xaxis=dict(visible=False), yaxis=dict(visible=False),
                  template="plotly_white", height=300)
fig.show()

## 7. Model evaluation

Report accuracy, precision, recall, F1-score, confusion matrix, and ROC-AUC. The positive class is benign (`1`) as encoded by the source dataset. For a real medical application, class definitions and error costs must be reviewed with domain experts.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test_np, verbose=0)
y_prob = model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

metrics = {
    "Accuracy": accuracy_score(y_test_np, y_pred),
    "Precision": precision_score(y_test_np, y_pred, zero_division=0),
    "Recall": recall_score(y_test_np, y_pred, zero_division=0),
    "F1-score": f1_score(y_test_np, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_np, y_prob),
    "Test loss": test_loss
}
display(pd.DataFrame([metrics]).T.rename(columns={0: "Score"}).style.format("{:.4f}"))
print(classification_report(y_test_np, y_pred, target_names=["malignant", "benign"]))

In [ ]:
cm = confusion_matrix(y_test_np, y_pred)
fig = px.imshow(cm, text_auto=True, color_continuous_scale="Blues",
                x=["Predicted malignant", "Predicted benign"],
                y=["Actual malignant", "Actual benign"],
                title="Confusion matrix")
fig.update_layout(template="plotly_white")
fig.show()

fpr, tpr, _ = roc_curve(y_test_np, y_prob)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name=f"MLP (AUC={metrics['ROC-AUC']:.3f})"))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", name="Random baseline",
                         line=dict(dash="dash")))
fig.update_layout(title="ROC curve", xaxis_title="False positive rate",
                  yaxis_title="True positive rate", template="plotly_white")
fig.show()

## 8. Experiment: compare activation functions and architectures

To satisfy the architecture/activation experiment, compare a compact and deeper network with ReLU, tanh, and ELU. Each candidate uses the same training/test split and preprocessing. Validation split and early stopping are retained. Results can vary slightly by hardware and TensorFlow version.

In [ ]:
experiments = [
    ("ReLU 64-32", "relu", (64, 32)),
    ("Tanh 64-32", "tanh", (64, 32)),
    ("ELU 64-32", "elu", (64, 32)),
    ("ReLU 32", "relu", (32,)),
    ("ReLU 128-64-32", "relu", (128, 64, 32)),
]
experiment_rows = []
for label, activation, hidden in experiments:
    candidate = build_mlp(X_train.shape[1], activation=activation, hidden=hidden)
    candidate.fit(X_train, y_train_np, validation_split=0.20, epochs=150,
                  batch_size=32, callbacks=[EarlyStopping(monitor="val_loss", patience=15,
                  restore_best_weights=True)], verbose=0)
    p = candidate.predict(X_test, verbose=0).ravel()
    pred = (p >= 0.5).astype(int)
    experiment_rows.append({
        "Model": label,
        "Parameters": candidate.count_params(),
        "Accuracy": accuracy_score(y_test_np, pred),
        "Precision": precision_score(y_test_np, pred, zero_division=0),
        "Recall": recall_score(y_test_np, pred, zero_division=0),
        "F1-score": f1_score(y_test_np, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test_np, p)
    })

experiment_df = pd.DataFrame(experiment_rows).sort_values("F1-score", ascending=False)
display(experiment_df)
fig = px.bar(experiment_df, x="Model", y=["Accuracy", "F1-score", "ROC-AUC"],
             barmode="group", title="Activation and architecture comparison",
             range_y=[0, 1], template="plotly_white")
fig.show()

## 9. Interpretation: overfitting and underfitting

- **Overfitting indicators:** training accuracy substantially higher than validation accuracy, or training loss continuing to decrease while validation loss rises.
- **Underfitting indicators:** both training and validation accuracy remain low, with both losses high.
- **Reasonable generalization:** training and validation curves remain close and validation loss does not diverge strongly.
- The test metrics estimate performance on held-out data. They do not establish clinical safety or suitability.

Use the actual learning curves and test scores above to write your observation after executing the notebook.

## 10. Self-learning questions and answers

**Q1. What is an MLP?**  
An MLP is a feed-forward artificial neural network containing an input layer, one or more fully connected hidden layers, and an output layer.

**Q2. Why is feature scaling important for an MLP?**  
Features on very different scales can lead to poorly conditioned optimization and slower or unstable learning. Standardization places features on a comparable scale.

**Q3. Why is ReLU used in hidden layers?**  
ReLU introduces non-linearity and is computationally inexpensive. It often supports efficient gradient-based training.

**Q4. Why is sigmoid used in the output layer?**  
For binary classification, sigmoid converts the output logit to a value between 0 and 1 that can be interpreted as the estimated probability of the positive class.

**Q5. Why use binary cross-entropy?**  
It is a standard loss for binary classification and penalizes disagreement between predicted probabilities and binary labels.

**Q6. What is the purpose of train/test splitting?**  
The training set is used to fit the model; the held-out test set estimates performance on unseen observations.

**Q7. What is data leakage?**  
Data leakage occurs when information from the test set influences training or preprocessing. Fitting the scaler on all observations is one example; fit preprocessing on training data only.

**Q8. How are Dense-layer parameters calculated?**  
For `n_in` inputs and `n_out` neurons, parameters equal `n_in × n_out` weights plus `n_out` biases.

**Q9. What is the difference between trainable and non-trainable parameters?**  
Trainable parameters are updated by the optimizer. Non-trainable parameters are stored in the model but are not updated during training. This model has no non-trainable parameters.

**Q10. What does a confusion matrix show?**  
It counts correct and incorrect predictions by actual and predicted class, revealing the types of classification errors.

**Q11. What is the difference between precision and recall?**  
Precision is the fraction of predicted positives that are truly positive. Recall is the fraction of actual positives that are detected.

**Q12. How can overfitting be reduced?**  
Use early stopping, dropout, regularization, a simpler architecture, more representative training data, or data augmentation where appropriate.

**Q13. Why compare activation functions and architectures?**  
Their capacity and optimization behavior can affect convergence, generalization, parameter count, and predictive performance.

**Q14. What is the role of an optimizer such as Adam?**  
An optimizer updates trainable weights and biases to minimize the chosen loss function.

**Q15. Why should this model not be used as a standalone medical diagnostic tool?**  
The exercise uses a limited benchmark dataset and does not include clinical validation, prospective evaluation, calibration assessment, fairness analysis, or regulatory review.

## 11. Conclusion

This lab implements a binary-classification MLP end to end: dataset inspection, train/test split, missing-value handling, feature scaling, architecture design, compilation, training, parameter accounting, evaluation, visualization, and architecture/activation comparison. After running all cells, use the displayed metrics and plots to write a concise conclusion specific to your execution.

**Submission reminder:** The lab sheet requests a filename containing your name, the last three register-number digits, and program number. Rename the deliverables accordingly before submission.